## Fine-Tuning LLaMA-2-7B using LoRA and QLoRA

In this notebook, we fine-tune the **LLaMA-2-7B** model on a conversational dataset using two parameter-efficient techniques:

1. **QLoRA** (Quantized LoRA) — loads the base model in 4-bit precision using `bitsandbytes` to reduce GPU memory usage.
2. **LoRA** (Low-Rank Adaptation) — injects trainable low-rank matrices into the attention layers so we only update a small fraction of the total parameters.
3. The model is trained using `SFTTrainer` from the `trl` library.
4. Finally, we run inference to test the fine-tuned model.

### Step 1: Install Dependencies

In [1]:
!pip -q install transformers
!pip -q install peft
!pip -q install accelerate
!pip -q install bitsandbytes
!pip -q install datasets
!pip -q install trl
!pip -q install scipy

### Step 2: Import Libraries

In [2]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer

### Step 3: Load Dataset

In [3]:
def load_training_data():
    dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")
    return dataset

dataset = load_training_data()
print(f"Dataset size: {len(dataset)}")
print(dataset[0]['text'][:300])

Dataset size: 1000
<s>[INST] What is the difference between supervised and unsupervised learning? [/INST] Supervised learning uses labeled data where the model learns to map inputs to outputs, while unsupervised learning finds patterns in unlabeled data without predefined outputs. </s>


### Step 4: QLoRA Configuration (4-bit Quantization)

In [4]:
def get_bnb_config():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=False
    )
    return bnb_config

### Step 5: Load Base Model and Tokenizer

In [5]:
def load_model_and_tokenizer(model_name, bnb_config):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )
    model.config.use_cache = False
    model.config.pretraining_tp = 1

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    return model, tokenizer

model_name = "meta-llama/Llama-2-7b-hf"
bnb_config = get_bnb_config()
model, tokenizer = load_model_and_tokenizer(model_name, bnb_config)

print(f"Model loaded: {model_name}")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Model loaded: meta-llama/Llama-2-7b-hf
GPU memory allocated: 3.86 GB


### Step 6: LoRA Configuration

In [6]:
def get_lora_config():
    lora_config = LoraConfig(
        r=64,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    return lora_config

model = prepare_model_for_kbit_training(model)
lora_config = get_lora_config()
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 33,554,432 || all params: 3,540,389,888 || trainable%: 0.9479


### Step 7: Training Arguments

In [7]:
def get_training_args(output_dir="./llama2-lora-finetuned"):
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=1,
        optim="paged_adamw_32bit",
        save_steps=50,
        logging_steps=25,
        learning_rate=2e-4,
        weight_decay=0.001,
        fp16=False,
        bf16=False,
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        group_by_length=True,
        lr_scheduler_type="cosine",
        report_to="none"
    )
    return training_args

### Step 8: Fine-Tune the Model

In [8]:
def train_model(model, dataset, lora_config, tokenizer, training_args):
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        peft_config=lora_config,
        dataset_text_field="text",
        max_seq_length=512,
        tokenizer=tokenizer,
        args=training_args,
        packing=False
    )
    trainer.train()
    return trainer

training_args = get_training_args()
trainer = train_model(model, dataset, lora_config, tokenizer, training_args)

Step,Training Loss
25,1.742800
50,1.521300
75,1.483100
100,1.461200
125,1.449700
150,1.438500
175,1.425300
200,1.419800
225,1.412600
250,1.408200


### Step 9: Save the Fine-Tuned Model

In [9]:
def save_model(trainer, save_path="./llama2-lora-finetuned"):
    trainer.model.save_pretrained(save_path)
    print(f"Model saved to {save_path}")

save_model(trainer)

Model saved to ./llama2-lora-finetuned


### Step 10: Inference

In [10]:
def run_inference(model, tokenizer, prompt, max_new_tokens=200):
    pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens
    )
    result = pipe(f"<s>[INST] {prompt} [/INST]")
    return result[0]['generated_text']

prompt = "What is the difference between LoRA and QLoRA?"
output = run_inference(model, tokenizer, prompt)
print(output)

<s>[INST] What is the difference between LoRA and QLoRA? [/INST] LoRA (Low-Rank Adaptation) is a technique that injects trainable low-rank matrices into transformer layers to fine-tune large language models efficiently. QLoRA extends this by loading the base model in 4-bit quantized format using NF4 quantization, which dramatically reduces GPU memory requirements while maintaining model quality. The key difference is that QLoRA uses quantization on top of LoRA, making it possible to fine-tune 7B+ parameter models on a single consumer GPU. </s>
